In [7]:
from langchain.chat_models import ChatOpenAI
from langchain.tools import BaseTool
from langchain.agents import initialize_agent, AgentType
from langchain.tools import DuckDuckGoSearchResults
from langchain.utilities import WikipediaAPIWrapper
from langchain.document_loaders import WebBaseLoader
from langchain.schema import SystemMessage
from typing import Type
from pydantic import BaseModel, Field
import os

llm = ChatOpenAI(
    temperature=0.1
)

# Wikipedia에서 정보 검색
class WikipediaSearchToolArgsSchema(BaseModel):
    query: str = Field(
        description="The topic to search for on Wikipedia."
    )

class WikipediaSearchTool(BaseTool):
    name = "WikipediaSearchTool"
    description = """
    Use this tool to search Wikipedia for information about a topic.
    It takes a search query as an argument.
    """
    args_schema: Type[WikipediaSearchToolArgsSchema] = WikipediaSearchToolArgsSchema

    def _run(self, query):
        wikipedia = WikipediaAPIWrapper()
        return wikipedia.run(query)

# 웹 검색 및 URL 탐색
class DuckDuckGoSearchToolArgsSchema(BaseModel):
    query: str = Field(
        description="The query to search for on DuckDuckGo."
    )

class DuckDuckGoSearchTool(BaseTool):
    name = "DuckDuckGoSearchTool"
    description = """
    Use this tool to search the web using DuckDuckGo.
    Use it when you need to find websites or additional
    information about a topic.
    The results may contain URLs that can be passed to
    WebsiteScrapingTool.
    """
    args_schema: Type[DuckDuckGoSearchToolArgsSchema] = DuckDuckGoSearchToolArgsSchema

    def _run(self, query):
        ddg = DuckDuckGoSearchResults()
        return ddg.run(query)

# 웹사이트에 들어가서 본문 내용 추출
class WebsiteScrapingToolArgsSchema(BaseModel):
    url: str = Field(
        description="The URL of the website to scrape."
    )

class WebsiteScrapingTool(BaseTool):
    name = "WebsiteScrapingTool"
    description = """
    Use this tool to extract the text content of a website.
    Pass a complete URL found from DuckDuckGo search results.

    Example:
    https://example.com/article
    """
    args_schema: Type[WebsiteScrapingToolArgsSchema] = WebsiteScrapingToolArgsSchema

    def _run(self, url):
        loader = WebBaseLoader(url)
        docs = loader.load()

        return "\n\n".join(
            doc.page_content
            for doc in docs
        )

# 최종 리서치를 research.txt에 저장
class SaveToFileToolArgsSchema(BaseModel):
    content: str = Field(
        description="The complete research content that should be saved."
    )

class SaveToFileTool(BaseTool):
    name = "SaveToFileTool"
    description = """
Use this tool to save the completed research report to research.txt.
You MUST use this tool after completing the research.
Pass the entire final research report as the content argument.
"""
    args_schema: Type[SaveToFileToolArgsSchema] = SaveToFileToolArgsSchema
    def _run(self, content):
        os.makedirs("./files/research", exist_ok=True)
        filename = "./files/research/research.txt"

        with open(filename, "w", encoding="utf-8",) as file:
            file.write(content)

        return f"Research successfully saved to {filename}"

agent = initialize_agent(
    llm=llm,
    verbose=True,
    agent=AgentType.OPENAI_FUNCTIONS,
    handle_parsing_errors=True,
    tools=[
        WikipediaSearchTool(),
        DuckDuckGoSearchTool(),
        WebsiteScrapingTool(),
        SaveToFileTool(),
    ],
    agent_kwargs={
        "system_message": SystemMessage(
            content="""
            You are a research agent.

            For every research task:
            - Search for relevant information.
            - Visit useful websites when necessary.
            - Write a complete research report.
            - ALWAYS invoke SaveToFileTool with the complete report.

            You MUST invoke SaveToFileTool before returning your final answer.
            Never finish a research task without saving the report.
            """
        )
    },
)

prompt = """
Research about the XZ backdoor.

First, search for information using Wikipedia or DuckDuckGo.

If you find useful websites through DuckDuckGo,
use WebsiteScrapingTool to visit and extract their content.

Use the information you collect to write a detailed research report.

Your research should include:
- What the XZ backdoor is
- How it was discovered
- How the attack worked
- What systems or versions were affected
- The security impact

Include the sources used in your research.

IMPORTANT:
After completing the research report, you MUST invoke
SaveToFileTool yourself.

Pass the entire completed research report to the
content argument of SaveToFileTool.

Do NOT tell the user to use SaveToFileTool.
You must actually call the tool yourself.

Do not finish until SaveToFileTool has successfully
saved the research.
"""

result = agent.invoke(prompt)

print(result)



> Entering new AgentExecutor chain...

Invoking: `DuckDuckGoSearchTool` with `{'query': 'XZ backdoor'}`




/Users/sonminseok/Desktop/web/full-stack-gpt/env/lib/python3.11/site-packages/langchain/utilities/duckduckgo_search.py:81: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/Users/sonminseok/Desktop/web/full-stack-gpt/env/lib/python3.11/site-packages/langchain/utilities/duckduckgo_search.py:82: UserWarning: backend='api' is deprecated, using backend='auto'
  results = ddgs.text(


[snippet: Jul 13, 2026 · Explore Luis Rodríguez's analysis of the XZ Backdoor attack: A stealthy SSH vulnerability exposed and …, title: XZ Backdoor: “That was a close one” | Xygeni, link: https://xygeni.io/blog/xz-backdoor-that-was-a-close-one/], [snippet: Jul 20, 2026 · 'Half a Second' chronicles the XZ Utils backdoor attempt—from a patient impostor maintainer to the half-second …, title: The XZ Backdoor Long Con Chronicled in ‘Half a Second’, link: https://fossforce.com/2026/07/the-xz-backdoor-long-con-chronicled-in-half-a-second/], [snippet: Feb 26, 2026 · A Microsoft engineer discovered a sophisticated state sponsored backdoor hidden in XZ Utils, a …, title: Internet Nearly Collapsed From XZ Backdoor Side Project, link: https://andrewbaker.ninja/2026/02/26/xz-utils-backdoor-how-one-engineer-saved-the-internet/], [snippet: Feb 27, 2026 · In late March 2024, the global cybersecurity and open-source software communities were confronted …, title: The Hacker Who Almost Stole the Intern